In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("anime.csv")
df

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266
...,...,...,...,...,...,...,...
12289,9316,Toushindai My Lover: Minami tai Mecha-Minami,Hentai,OVA,1,4.15,211
12290,5543,Under World,Hentai,OVA,1,4.28,183
12291,5621,Violence Gekiga David no Hoshi,Hentai,OVA,4,4.88,219
12292,6133,Violence Gekiga Shin David no Hoshi: Inma Dens...,Hentai,OVA,1,4.98,175


In [2]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [4]:
df['genre'] = df['genre'].fillna('Unknown')
df['type'] = df['type'].fillna('Unknown')
median_rating = df['rating'].median()
df['rating'] = df['rating'].fillna(median_rating)
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
print("Missing values handled and 'episodes' column converted to numeric.")

Missing values handled and 'episodes' column converted to numeric.


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12294 non-null  object 
 3   type      12294 non-null  object 
 4   episodes  11954 non-null  float64
 5   rating    12294 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(2), int64(2), object(3)
memory usage: 672.5+ KB


In [6]:
print("Descriptive statistics for numerical columns:")
print(df.describe())
print("\nUnique values count for 'genre' column:")
print(df['genre'].nunique())
print("\nUnique values count for 'type' column:")
print(df['type'].nunique())

Descriptive statistics for numerical columns:
           anime_id      episodes        rating       members
count  12294.000000  11954.000000  12294.000000  1.229400e+04
mean   14058.221653     12.382550      6.475700  1.807134e+04
std    11455.294701     46.865352      1.017179  5.482068e+04
min        1.000000      1.000000      1.670000  5.000000e+00
25%     3484.250000      1.000000      5.900000  2.250000e+02
50%    10260.500000      2.000000      6.570000  1.550000e+03
75%    24794.500000     12.000000      7.170000  9.437000e+03
max    34527.000000   1818.000000     10.000000  1.013917e+06

Unique values count for 'genre' column:
3265

Unique values count for 'type' column:
7


# Data Preprocessing & Feature extraction

In [7]:
df_similarity = df[['genre', 'rating', 'members', 'episodes', 'type']].copy()
print("Created df_similarity with selected columns:")
df_similarity.head()

Created df_similarity with selected columns:


,genre,rating,members,episodes,type
0,"Drama, Romance, School, Supernatural",9.37,200630,1.0,Movie
1,"Action, Adventure, Drama, Fantasy, Magic, Mili...",9.26,793665,64.0,TV
2,"Action, Comedy, Historical, Parody, Samurai, S...",9.25,114262,51.0,TV
3,"Sci-Fi, Thriller",9.17,673572,24.0,TV
4,"Action, Comedy, Historical, Parody, Samurai, S...",9.16,151266,51.0,TV


In [8]:
from sklearn.preprocessing import MultiLabelBinarizer

# Handle the 'genre' column by splitting multiple genres and then one-hot encoding
# Fill NaN values in 'genre' with an empty string list before processing
# This ensures that entries with 'Unknown' (previously filled) are treated as a single category, and actual NaNs are handled.
df_similarity['genre'] = df_similarity['genre'].apply(lambda x: [g.strip() for g in str(x).split(',') if g.strip()])

mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(df_similarity['genre'])
genre_df = pd.DataFrame(genre_encoded, columns=mlb.classes_, index=df_similarity.index)

df_similarity = pd.concat([df_similarity.drop('genre', axis=1), genre_df], axis=1)

print("Genre column encoded successfully.")
df_similarity.head()

Genre column encoded successfully.


,rating,members,episodes,type,Action,Adventure,Cars,Comedy,Dementia,Demons,...,Slice of Life,Space,Sports,Super Power,Supernatural,Thriller,Unknown,Vampire,Yaoi,Yuri
0,9.37,200630,1.0,Movie,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1,9.26,793665,64.0,TV,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,9.25,114262,51.0,TV,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,9.17,673572,24.0,TV,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,9.16,151266,51.0,TV,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
df_similarity = pd.get_dummies(df_similarity, columns=['type'], prefix='type')
print("Type column encoded successfully.")
df_similarity.head()

Type column encoded successfully.


,rating,members,episodes,Action,Adventure,Cars,Comedy,Dementia,Demons,Drama,...,Vampire,Yaoi,Yuri,type_Movie,type_Music,type_ONA,type_OVA,type_Special,type_TV,type_Unknown
0,9.37,200630,1.0,0,0,0,0,0,0,1,...,0,0,0,True,False,False,False,False,False,False
1,9.26,793665,64.0,1,1,0,0,0,0,1,...,0,0,0,False,False,False,False,False,True,False
2,9.25,114262,51.0,1,0,0,1,0,0,0,...,0,0,0,False,False,False,False,False,True,False
3,9.17,673572,24.0,0,0,0,0,0,0,0,...,0,0,0,False,False,False,False,False,True,False
4,9.16,151266,51.0,1,0,0,1,0,0,0,...,0,0,0,False,False,False,False,False,True,False


In [10]:
from sklearn.preprocessing import StandardScaler

# Identify numerical columns to normalize
numerical_cols = ['rating', 'members', 'episodes']

# Initialize StandardScaler
scaler = StandardScaler()

# Handle NaN values in 'episodes' column by filling with 0 before scaling, as StandardScaler cannot handle NaNs.
# This is a temporary measure for scaling; the NaNs represent non-numeric original values.
df_similarity['episodes'] = df_similarity['episodes'].fillna(0)

# Apply scaling to the numerical columns
df_similarity[numerical_cols] = scaler.fit_transform(df_similarity[numerical_cols])

print("Numerical columns normalized successfully.")
df_similarity.head()

Numerical columns normalized successfully.


,rating,members,episodes,Action,Adventure,Cars,Comedy,Dementia,Demons,Drama,...,Vampire,Yaoi,Yuri,type_Movie,type_Music,type_ONA,type_OVA,type_Special,type_TV,type_Unknown
0,2.845534,3.330241,-0.238677,0,0,0,0,0,0,1,...,0,0,0,True,False,False,False,False,False,False
1,2.737388,14.148406,1.123326,1,1,0,0,0,0,1,...,0,0,0,False,False,False,False,False,True,False
2,2.727556,1.754713,0.842278,1,0,0,1,0,0,0,...,0,0,0,False,False,False,False,False,True,False
3,2.648904,11.957666,0.258562,0,0,0,0,0,0,0,...,0,0,0,False,False,False,False,False,True,False
4,2.639073,2.429742,0.842278,1,0,0,1,0,0,0,...,0,0,0,False,False,False,False,False,True,False


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim_matrix = cosine_similarity(df_similarity)

print("Shape of cosine similarity matrix:", cosine_sim_matrix.shape)

Shape of cosine similarity matrix: (12294, 12294)


In [12]:
def get_recommendations(anime_title, cosine_sim_matrix, df, top_n=10, similarity_threshold=0.0):
    if anime_title not in df['name'].values:
        print(f"Anime '{anime_title}' not found in the database.")
        return []

    # 2. Find the index of the anime_title in the df DataFrame
    idx = df[df['name'] == anime_title].index[0]

    # 3. Retrieve the similarity scores for the target anime
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # 5. Sort this list in descending order based on the similarity score
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # 6. Filter out the target anime itself
    # Get a list of indices of all animes that are not the target anime
    anime_indices = [i[0] for i in sim_scores if i[0] != idx]

    # 7. Apply the similarity_threshold if provided
    filtered_sim_scores = [score for score in sim_scores if score[1] >= similarity_threshold and score[0] != idx]

    # 8. Select the top top_n recommendations from the filtered list
    top_recommendations = filtered_sim_scores[0:top_n]

    # 9. Map the indices of the recommended animes back to their names
    recommended_animes = []
    for i, score in top_recommendations:
        recommended_animes.append({'name': df.loc[i, 'name'], 'similarity_score': score})

    # 10. Return a list of recommended anime titles along with their similarity scores
    return recommended_animes

print("Recommendation function 'get_recommendations' defined.")

Recommendation function 'get_recommendations' defined.


In [13]:
# Example usage:
example_anime = 'Steins;Gate'
recommendations = get_recommendations(example_anime, cosine_sim_matrix, df)
print(f"\nTop 10 recommendations for '{example_anime}':")
for anime in recommendations:
    print(f"- {anime['name']} (Similarity: {anime['similarity_score']:.4f})")


Top 10 recommendations for 'Steins;Gate':
- Death Note (Similarity: 0.9854)
- Code Geass: Hangyaku no Lelouch (Similarity: 0.9816)
- Shingeki no Kyojin (Similarity: 0.9798)
- Psycho-Pass (Similarity: 0.9791)
- Durarara!! (Similarity: 0.9784)
- Tengen Toppa Gurren Lagann (Similarity: 0.9781)
- Toradora! (Similarity: 0.9775)
- Angel Beats! (Similarity: 0.9763)
- Mirai Nikki (TV) (Similarity: 0.9760)
- Zankyou no Terror (Similarity: 0.9760)


In [14]:
target_anime = 'Steins;Gate'
thresholds_to_test = [0.7, 0.8, 0.9, 0.95]

print(f"Experimenting with recommendation thresholds for '{target_anime}':\n")

for threshold in thresholds_to_test:
    print(f"--- Recommendations for similarity_threshold = {threshold} ---")
    recommendations_with_threshold = get_recommendations(target_anime, cosine_sim_matrix, df, top_n=20, similarity_threshold=threshold)

    if not recommendations_with_threshold:
        print("No recommendations found for this threshold.")
    else:
        for anime in recommendations_with_threshold:
            print(f"- {anime['name']} (Similarity: {anime['similarity_score']:.4f})")
    print("\n")

Experimenting with recommendation thresholds for 'Steins;Gate':

--- Recommendations for similarity_threshold = 0.7 ---
- Death Note (Similarity: 0.9854)
- Code Geass: Hangyaku no Lelouch (Similarity: 0.9816)
- Shingeki no Kyojin (Similarity: 0.9798)
- Psycho-Pass (Similarity: 0.9791)
- Durarara!! (Similarity: 0.9784)
- Tengen Toppa Gurren Lagann (Similarity: 0.9781)
- Toradora! (Similarity: 0.9775)
- Angel Beats! (Similarity: 0.9763)
- Mirai Nikki (TV) (Similarity: 0.9760)
- Zankyou no Terror (Similarity: 0.9760)
- Deadman Wonderland (Similarity: 0.9757)
- Fullmetal Alchemist: Brotherhood (Similarity: 0.9753)
- Sword Art Online (Similarity: 0.9749)
- Mahou Shoujo Madoka★Magica (Similarity: 0.9747)
- Code Geass: Hangyaku no Lelouch R2 (Similarity: 0.9745)
- Guilty Crown (Similarity: 0.9727)
- Another (Similarity: 0.9726)
- Darker than Black: Kuro no Keiyakusha (Similarity: 0.9723)
- Akame ga Kill! (Similarity: 0.9723)
- Ano Hi Mita Hana no Namae wo Bokutachi wa Mada Shiranai. (Similari

# Interview Questions